### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [7]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-3.5-flash-lite")
response = model.invoke("Why do parrots talk? short answer")
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': 'In the wild, parrots are social animals that mimic sounds to bond with their flock, identify family members, and defend territory. In captivity, they use human speech for the same reason: **to communicate, build relationships, and get attention** from their human "flock."', 'extras': {'signature': 'EmAKXgFpFH0T5xt3i1wDrzwNZHyJK9ABg18wSnqBFiAMKuWR8w4GnqLkcw1GO8k1Wepuw0mZ+0QtFDmFWLYmkoTL4VgVUPcibHqgnF2ps7Sxv2RtGk+YPxITrjisTfOMz9s='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11288-aa76-7942-9d06-59c77dd5ed26-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 55, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}})

In [8]:
# two ways to add set of tools to a model:
#   1. bind_tools: binds a set of tools to a model(see below)
#   2.after creating an agent add a tool list as an argument to the agent.
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '7z3qd8gnm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.071765799, 'completion_tokens_details': None, 'prompt_time': 0.02289885, 'prompt_tokens_details': None, 'queue_time': 0.039330918, 'total_time': 0.094664649}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8da23ae6c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a11275-b493-7ac2-b1e0-1caa05bcfa1d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '7z3qd8gnm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's currently sunny in Boston.


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_171080': 'EmAKXgFpFH0TV616Wwn3pC5ljLhKrlRIwacWRaV0Uao++ypVaQ+BrmBfx6ZcSZqm5JvXRqeMKT8bxBSeGcjbg4pMAO7+2uus4YBUrt+X7RAsdhvDIrcaMTxxlVpPVKz/bBE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11288-e6b3-7361-ad07-0becdbfa2bfe-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_171080', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='call_171080')]